# Pipeline orchestrator

Single notebook to drive every stage. Each section is **flag-gated** at the
top — flip booleans on/off rather than commenting code in and out. Per-run
config tweaks happen in Python (mutating the `cfg` object) — no YAML edits.

**v0.9.0 update.** Three new knobs land here in addition to the original
stage toggles:

- `TARGET_SICKLE_FRAC` — gate `labels.csv` to a target sickle fraction
  (e.g. `0.10`) before fold construction. `None` = no gating.
- `FOLD_STRATEGY` — `"balanced"` (default, FOV-grouped greedy bin-packer)
  or `"stratified"` (the original sklearn-backed splitter; use this to
  reproduce the `ablation_20260516_003426` numbers).
- `RUN_MAE_PRETRAIN` + `MAE_PRETRAIN_CONFIG` — kick off the long
  single-GPU MAE continuation pretrain. Pair with
  `USE_MAE_CKPT_IN_ABLATION = True` to feed the produced checkpoint into
  every MAE-variant ablation row.

Suggested first pass for a fresh corpus rebuild + ablation:

1. `RUN_PREDICT = True`, `RUN_INSTANCES = True`, `RUN_CROPS = True` —
   rebuild `cells.parquet` against your chosen labels CSV.
2. `RUN_MAE_PRETRAIN = True` with `MAE_PRETRAIN_CONFIG = "configs/pretrain_mae_long.yaml"`
   to produce a fresh long-MAE checkpoint.
3. `USE_MAE_CKPT_IN_ABLATION = True`, `RUN_ABLATION = True` — runs the
   full PIPELINE_PLAN §4 table at the chosen sickle prevalence.
4. Inspect per-run wall-clock via the **duration summary** cell at the
   bottom of the notebook — single-GPU vs DDP comparisons can be done
   later by re-running the MAE pretrain under `--devices N --strategy ddp`
   and diffing the two `duration.json` files.

Re-running with a stage's flag = `False` skips that stage (uses what's on disk).

In [1]:
# Fallback if `pip install -e .` hasn't been run.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sickling.rbc_classification.py_modules.config import load_config
from sickling.rbc_classification.py_modules.engineering.seed import seed_everything

# ===========================================================================
# STAGE TOGGLES — flip per run
# ===========================================================================
RUN_PREDICT         = False    # Bulk-run U-Net over to_be_labeled/ -> unet_predictions/ + raw_images/
RUN_INSTANCES       = False    # Stage 2 watershed for every FOV in unet_predictions/
RUN_CROPS           = False    # Stage 3 — rebuild cells.parquet against the chosen labels CSV
RUN_SMOKE_FIT       = False    # Quick sanity run for a single variant
RUN_MAE_PRETRAIN    = False    # Long single-GPU MAE continuation pretrain (Model C)
RUN_ABLATION        = True     # Full ablation table
RUN_FIGURES         = True     # Re-render SVGs from any cached report.json

# ===========================================================================
# RUN-WIDE CONFIG (Python overrides — no YAML edits needed)
# ===========================================================================
USE_TRIMMED_LABELS = False   # True -> labels/labels_trimmed.csv ; False -> labels/labels.csv
FAST_MODE          = False   # smaller epoch budget + batch size for quick iteration

MODEL_PATH         = Path.cwd() / "models" / "unet_fold_2_best.pth"
TO_BE_LABELED_DIR  = Path.cwd() / "to_be_labeled"

# --- v0.9.0 — natural-prevalence gating + balanced folds --------------------
# Set TARGET_SICKLE_FRAC to None to keep the original sickle-enriched corpus;
# set to 0.10 to mimic the natural prevalence documented in PIPELINE_PLAN.md.
TARGET_SICKLE_FRAC = 0.10 # or None
# 'balanced' = FOV-grouped greedy bin-packer (fixes fold-4 outlier);
# 'stratified' = original sklearn splitter (use to reproduce 20260516 numbers).
FOLD_STRATEGY      = "balanced" # "stratified" or "balanced"

# --- v0.9.0 — long MAE pretrain --------------------------------------------
# Config used by the MAE pretrain stage. `pretrain_mae.yaml` = the original
# 300-epoch schedule; `pretrain_mae_long.yaml` = the 1600-epoch follow-up.
MAE_PRETRAIN_CONFIG       = "configs/pretrain_mae_long.yaml"
MAE_PRETRAIN_DEVICES      = 1
MAE_PRETRAIN_STRATEGY     = "auto"   # set to "ddp" with devices >= 2 for the DDP comparison
# Inject the latest pretrain checkpoint into every ablation row whose
# variant is 'mae' or whose image_variant is 'mae'. When False, the MAE
# rows use the timm-registry MAE weights (original behaviour).
USE_MAE_CKPT_IN_ABLATION  = True

# --- ablation crossing -----------------------------------------------------
ABLATION_SEEDS = (42, 43, 44)        # tuples of ints; e.g. (42, 43, 44) for 3-seed runs
ABLATION_FOLDS = (0, 1, 2, 3, 4)     # tuples of ints; e.g. (0, 1, 2, 3, 4) for full k-fold

SMOKE_FIT_VARIANT       = "multimodal"   # one of: dinov2_frozen | timm_vit | mae | multimodal
SMOKE_FIT_IMAGE_VARIANT = "dinov2_frozen"

# ---------------------------------------------------------------------------
cfg = load_config()
if USE_TRIMMED_LABELS:
    cfg.paths.labels_csv = Path("labels/labels_trimmed.csv")
if FAST_MODE:
    cfg.training.precision = "32-true"
    cfg.training.batch_size = 32
    cfg.training.max_epochs = 5
    cfg.training.warmup_epochs = 1
    cfg.training.num_workers = 0
    cfg.validation.bootstrap_resamples = 200

# v0.9.0 — apply the prevalence-gate + fold-strategy knobs to the live cfg.
cfg.validation.target_sickle_frac = TARGET_SICKLE_FRAC
cfg.validation.fold_strategy = FOLD_STRATEGY

seed_everything(cfg.project.seed)

paths = cfg.paths.resolved()
print("labels_csv          :", paths.labels_csv)
print("unet_predictions    :", paths.unet_predictions)
print("crops               :", paths.crops)
print("fast mode           :", FAST_MODE,
      "| batch_size:", cfg.training.batch_size,
      "| max_epochs:", cfg.training.max_epochs,
      "| precision:", cfg.training.precision)
print("trimmed labels      :", USE_TRIMMED_LABELS)
print("target_sickle_frac  :", cfg.validation.target_sickle_frac)
print("fold_strategy       :", cfg.validation.fold_strategy)
print("MAE pretrain config :", MAE_PRETRAIN_CONFIG, f"(devices={MAE_PRETRAIN_DEVICES}, strategy={MAE_PRETRAIN_STRATEGY})")
print("inject MAE ckpt     :", USE_MAE_CKPT_IN_ABLATION)

labels_csv          : E:\utku g leica\rbc-class\labels\labels.csv
unet_predictions    : E:\utku g leica\rbc-class\unet_predictions
crops               : E:\utku g leica\rbc-class\crops
fast mode           : False | batch_size: 64 | max_epochs: 50 | precision: bf16-mixed
trimmed labels      : False
target_sickle_frac  : 0.1
fold_strategy       : balanced
MAE pretrain config : configs/pretrain_mae_long.yaml (devices=1, strategy=auto)
inject MAE ckpt     : True


## Stage 1 — bulk U-Net prediction

Reads every image in `to_be_labeled/`, writes `unet_predictions/PRED_<stem>.h5`,
and copies the raw into `raw_images/`. Idempotent — files already on disk are skipped.

In [3]:
if RUN_PREDICT:
    from sickling.rbc_classification.py_modules.stage1_unet import bulk_predict
    bulk_predict(cfg, input_dir=TO_BE_LABELED_DIR, model_path=MODEL_PATH)
else:
    print("skip (RUN_PREDICT=False)")

skip (RUN_PREDICT=False)


## Stage 2 — instance segmentation

Walks every `unet_predictions/*.h5` and writes `instances/<stem>_instances.h5`.
Pass `qa=True` to render QA panels under `figures/` for visual sanity.

In [4]:
if RUN_INSTANCES:
    from sickling.rbc_classification.py_modules.stage2_instances.cli import run_stage2
    stats = run_stage2(cfg, qa=False)
    display(stats)
else:
    print("skip (RUN_INSTANCES=False)")

skip (RUN_INSTANCES=False)


## Stage 3 — crop extraction + cells.parquet

Uses `cfg.paths.labels_csv` (trimmed or full, set above). Rebuilds `cells.parquet`
from scratch — old labels not in the chosen CSV get dropped.

In [5]:
if RUN_CROPS:
    from sickling.rbc_classification.py_modules.stage3_crops.cli import run_stage3
    cells_df = run_stage3(cfg)
else:
    from sickling.rbc_classification.py_modules.io.parquet import read_cells
    cells_df = read_cells(paths.root / cfg.paths.cells_parquet)

labeled = cells_df[cells_df['has_label'] == True]
print(f"cells.parquet: {len(cells_df)} rows ({len(labeled)} labeled)")
print("label counts:\n", labeled['label'].value_counts() if not labeled.empty else "(none)")
print("FOVs labeled :", labeled['source_image'].nunique() if not labeled.empty else 0)

cells.parquet: 84895 rows (1851 labeled)
label counts:
 label
non_sickle    1108
sickle         713
ambiguous       30
Name: count, dtype: int64
FOVs labeled : 259


## Quick single-model sanity run

Optional — trains one variant for the current `max_epochs` to confirm the loop
works end-to-end before kicking off the longer ablation.

In [6]:
if RUN_SMOKE_FIT:
    if SMOKE_FIT_VARIANT == "multimodal":
        from sickling.rbc_classification.py_modules.stage5_multimodal.cli import run_multimodal_finetune
        out = run_multimodal_finetune(
            cfg, image_variant=SMOKE_FIT_IMAGE_VARIANT, fold=0
        )
    else:
        from sickling.rbc_classification.py_modules.stage4_repr.cli import run_finetune
        out = run_finetune(cfg, variant=SMOKE_FIT_VARIANT, fold=0)
    print(out)
else:
    print("skip (RUN_SMOKE_FIT=False)")

skip (RUN_SMOKE_FIT=False)


## Stage 4 — long MAE continuation pretrain (Model C)

Single-GPU by default; the schedule lives in `configs/pretrain_mae_long.yaml`
(800 epochs, batch 192, grad-accum 2 → effective 384, 40-epoch warmup).
The `DurationCallback` writes `duration.json` next to the checkpoints so
this run is directly comparable to a later DDP run.

Outputs:
- `checkpoints/pretrain_mae/` — Lightning checkpoints (`save_top_k`).
- `checkpoints/pretrain_mae/duration.json` — wall-clock + imgs/sec summary.

In [7]:
if RUN_MAE_PRETRAIN:
    from sickling.rbc_classification.py_modules.config import load_config as _load_config
    from sickling.rbc_classification.py_modules.stage4_repr.cli import run_pretrain_mae

    # Re-load cfg with the long-MAE override on top of base.yaml so we don't
    # have to mutate the live `cfg` (which is sized for ablation, not pretrain).
    mae_cfg = _load_config(MAE_PRETRAIN_CONFIG)
    # Carry over the global seed + paths from the live cfg so any user-side
    # path overrides (trimmed labels, etc.) propagate.
    mae_cfg.project.seed = cfg.project.seed
    mae_cfg.paths = cfg.paths

    out = run_pretrain_mae(
        mae_cfg,
        devices=MAE_PRETRAIN_DEVICES,
        strategy=MAE_PRETRAIN_STRATEGY,
    )
    MAE_CKPT_PATH = Path(out.get("best_checkpoint", ""))
    print(f"MAE pretrain done. best_ckpt = {MAE_CKPT_PATH}")
else:
    # Fall back to the latest pretrain checkpoint on disk, if any.
    _pre_dir = paths.checkpoints / "pretrain_mae"
    _ckpts = sorted(_pre_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime) if _pre_dir.exists() else []
    MAE_CKPT_PATH = _ckpts[-1] if _ckpts else None
    print("skip (RUN_MAE_PRETRAIN=False) — latest existing ckpt:", MAE_CKPT_PATH)

skip (RUN_MAE_PRETRAIN=False) — latest existing ckpt: None


## Ablation table

Runs the PIPELINE_PLAN §4 table across `ABLATION_SEEDS × ABLATION_FOLDS`.
Crash-resumable: results stream to `figures/ablation/<timestamp>/raw_results.json`
after every cell.

Edit `rows = ...` below to subset the table for a faster sweep.

## Display the latest ablation table inline

In [8]:
from IPython.display import Markdown, display

ablation_root = paths.figures / "ablation"
candidates = sorted(ablation_root.glob("*/table_markdown.md"), key=lambda p: p.stat().st_mtime)
if candidates:
    latest = candidates[-1]
    print("showing:", latest)
    display(Markdown(latest.read_text(encoding="utf-8")))
else:
    print("No ablation tables on disk yet — run with RUN_ABLATION=True.")

No ablation tables on disk yet — run with RUN_ABLATION=True.


### Per-fold composition preview

Mirrors the gate + fold-strategy applied by  / 
internally, so you see the per-fold sickle / non_sickle counts BEFORE the
ablation starts. Flag any fold with  and re-tune
 or  before committing 100+ runs.

In [9]:
# v0.9.1 — per-fold class composition preview.
# Runs BEFORE the ablation cell so you can spot fold-skew problems early
# (e.g. the rare-minority-clumping pathology from ablation_20260517_154137).
from sickling.rbc_classification.py_modules.data.crop_dataset import labeled_subset
from sickling.rbc_classification.py_modules.eval.splits import fold_diagnostics, make_kfold_splits
from sickling.rbc_classification.py_modules.io.labels import gate_labels_to_prevalence
from sickling.rbc_classification.py_modules.io.parquet import read_cells

_preview_cells = read_cells(paths.root / cfg.paths.cells_parquet)
_preview_lab = labeled_subset(_preview_cells)
if cfg.validation.target_sickle_frac is not None:
    _preview_lab, _gate_stats = gate_labels_to_prevalence(
        _preview_lab,
        target_sickle_frac=cfg.validation.target_sickle_frac,
        seed=cfg.validation.gate_seed,
    )
    print("[gate]", _gate_stats)
_preview_splits = make_kfold_splits(
    _preview_lab,
    n_splits=cfg.validation.cv_folds,
    seed=cfg.project.seed,
    strategy=cfg.validation.fold_strategy,
)
_preview_diag = fold_diagnostics(_preview_lab, _preview_splits)
_total_sickle = int(_preview_diag["n_sickle_val"].sum())
_total_non = int(_preview_diag["n_non_sickle_val"].sum())
_preview_diag["% sickle"] = (
    100 * _preview_diag["n_sickle_val"] /
    (_preview_diag["n_sickle_val"] + _preview_diag["n_non_sickle_val"]).clip(lower=1)
).round(1)
print(f"Strategy: {cfg.validation.fold_strategy} | target_sickle_frac: {cfg.validation.target_sickle_frac}")
print(f"Total labeled-modelled: n_sickle={_total_sickle} n_non_sickle={_total_non}")
display(_preview_diag)

# Friendly health check — every fold should hold at least one sickle.
_min_sickle = int(_preview_diag["n_sickle_val"].min())
if _min_sickle == 0:
    print("WARNING: at least one fold has zero sickle val cells — ablation MCC will be undefined there.")
elif _min_sickle < 5:
    print(f"NOTE: smallest fold has only {_min_sickle} sickle val cells — bootstrap CIs will be wide.")

[gate] {'n_sickle_in': 713, 'n_non_sickle_in': 1108, 'n_sickle_kept': 123, 'n_non_sickle_kept': 1108, 'n_dropped': 590, 'achieved_frac': 0.09991876523151909, 'target_frac': 0.1, 'policy': 'drop_excess_majority', 'seed': 42}
Strategy: balanced | target_sickle_frac: 0.1
Total labeled-modelled: n_sickle=123 n_non_sickle=1108


,fold,n_val,n_train,n_sickle_val,n_non_sickle_val,n_fovs_val,% sickle
0,0,224,1007,27,197,18,12.1
1,1,250,981,27,223,24,10.8
2,2,222,1009,27,195,20,12.2
3,3,273,958,27,246,26,9.9
4,4,262,969,15,247,15,5.7


In [ ]:
if RUN_ABLATION:
    from sickling.rbc_classification.py_modules.ablation import (
        DEFAULT_ABLATION, aggregate_results, run_ablation_table, write_tables,
    )
    from sickling.rbc_classification.py_modules.ablation import runner as _runner  # for monkey-patch below

    # Use DEFAULT_ABLATION for the full §4 table (now 9 rows incl. the new
    # per-tower mask-zero row), or pick a subset for speed:
    rows = DEFAULT_ABLATION
    # rows = [r for r in DEFAULT_ABLATION if r.variant in ('multimodal', 'dinov2_frozen')]

    # ---- v0.9.0: optionally inject the long-MAE checkpoint into MAE rows ---
    if USE_MAE_CKPT_IN_ABLATION and MAE_CKPT_PATH is not None and Path(MAE_CKPT_PATH).exists():
        _orig_train_one = _runner._train_one
        _orig_eval_one  = _runner._eval_one

        def _train_one_with_mae(cfg, row, seed, fold, synth_labels):
            from sickling.rbc_classification.py_modules.stage4_repr.cli import run_finetune as _rf
            from sickling.rbc_classification.py_modules.stage5_multimodal.cli import run_multimodal_finetune as _rmf
            cfg2 = _runner._apply_overrides(cfg, row.overrides)
            _runner.seed_everything(seed)
            try:
                if row.variant == "multimodal" and row.image_variant == "mae":
                    return _rmf(
                        cfg2, image_variant="mae", fold=fold,
                        mae_init_ckpt=Path(MAE_CKPT_PATH),
                        synth_labels=synth_labels,
                        use_image=row.use_image, use_morphology=row.use_morphology,
                        zero_mask_channels=row.zero_mask_channels,
                        zero_image_masks_only=row.zero_image_masks_only,
                    )
                if row.variant == "mae":
                    return _rf(
                        cfg2, variant="mae", fold=fold,
                        mae_init_ckpt=Path(MAE_CKPT_PATH),
                        synth_labels=synth_labels,
                        zero_mask_channels=row.zero_mask_channels or row.zero_image_masks_only,
                    )
                return _orig_train_one(cfg, row, seed, fold, synth_labels)
            finally:
                _runner._finish_wandb_run()

        _runner._train_one = _train_one_with_mae
        print(f"[ablation] injecting MAE checkpoint {MAE_CKPT_PATH} into MAE rows")
    elif USE_MAE_CKPT_IN_ABLATION:
        print("[ablation] USE_MAE_CKPT_IN_ABLATION=True but no checkpoint on disk; MAE rows will use timm weights.")

    results = run_ablation_table(
        cfg,
        rows=rows,
        seeds=ABLATION_SEEDS,
        folds=ABLATION_FOLDS,
    )
    agg = aggregate_results(results)
    # Latest output dir created by the runner.
    ablation_root = paths.figures / "ablation"
    out_dir = max(ablation_root.iterdir(), key=lambda p: p.stat().st_mtime)
    paths_out = write_tables(agg, out_dir, title="Ablation results")
    print("\nwrote:", paths_out)
else:
    print("skip (RUN_ABLATION=False)")

## Re-render figures

Re-renders SVGs from every `report.json` under `figures/eval/`. No retraining.

In [ ]:
if RUN_FIGURES:
    from sickling.rbc_classification.py_modules.eval.cli import run_figures
    run_figures(cfg)
else:
    print("skip (RUN_FIGURES=False)")

## Run-duration summary

Aggregates every `duration.json` written by `DurationCallback` (Stage 4 /
Stage 5 / MAE pretrain) into a single table for quick reading. Use this
when comparing single-GPU vs DDP runs side-by-side.

In [ ]:
import json as _json
from pathlib import Path as _Path
import pandas as _pd

_rows = []
for _p in sorted((paths.checkpoints).rglob("duration.json")):
    try:
        _d = _json.loads(_p.read_text())
    except Exception:
        continue
    _rows.append({
        "run_name": _d.get("run_name", _p.parent.name),
        "device": _d.get("device_name"),
        "n_devices": _d.get("n_devices"),
        "strategy": _d.get("strategy"),
        "precision": _d.get("precision"),
        "epochs": _d.get("completed_epochs"),
        "fit_seconds": round(_d.get("fit_seconds", 0.0), 1),
        "imgs/sec": round(_d.get("imgs_per_second_mean", 0.0), 1),
        "samples_seen": _d.get("samples_seen_total"),
    })

if _rows:
    _df = _pd.DataFrame(_rows).sort_values("run_name").reset_index(drop=True)
    display(_df)
else:
    print("No duration.json files yet — kick off a training run first.")